# Problema das Marginais Quânticas (Quantum Marginal Problem - QMP) via Programação Semidefinida (SDP)

Este notebook é um guia de referência e implementação fundamental para as primitivas de mecânica quântica e informação quântica utilizadas na resolução do **Problema das Marginais Quânticas (QMP)**.

### Objetivos Teóricos:
1. **Representação de Estados Densidade**: Definir vetores de estado kets, projetores e validar os postulados de matrizes densidade (Hermiticiade, traço unitário e positividade semidefinida $\rho \ge 0$).
2. **Espaço Composto e Traço Parcial**: Operações de produto tensorial (estado GHZ) e eliminação de subsistemas via traço parcial com contração tensorial vetorizada (`np.einsum`).
3. **Dualidade de Hilbert-Schmidt e Embedding**: Construção de bases ortonormais em $\text{Herm}(d)$ e representação de observáveis locais no espaço global.
4. **Formulação do SDP Cônico**: Conversão de restrições marginais em igualdades afins $\text{Tr}(M_i \rho) = b_i$ para a biblioteca `SDPLab` e resolução via **CLARABEL**.


## 1. Importação de Primitivas Fundamentais e Bibliotecas

Carregamento do módulo `funcoes_fundamentais.py` contendo funções primitivas para álgebras quânticas, geradores de base e solvers SDP.


In [1]:
from funcoes_fundamentais import *
import pandas as pd
import numpy as np


### 1.1 Instalação Automatizada de Pacotes

Instalação dos pacotes de Otimização Convexa / SDP (`spacecore==0.4.2`, `sdplab`, `clarabel`).


In [ ]:
# Instalação das dependências necessárias no ambiente
# Nota: spacecore==0.4.2 é recomendado para total compatibilidade com sdplab==0.0.1
! pip install -q numpy matplotlib scipy "spacecore==0.4.2" sdplab cvxpy clarabel scs jax

### 1.2 Configuração de Módulos Auxiliares

Importação de suporte a anotações de tipo avançadas (`annotations`) e gerenciamento do ambiente.


In [3]:
from __future__ import annotations

import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida 
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

from funcoes_fundamentais import zero, one, ketbra, verifica_densidade, verify_marginals, base_hermitiana, ghz, partial_trace, tensor, embed_operator, solve_qmp, build_qmp_sdp, parse_subsystem_key, basis
print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

Versão SDPLab:    0.0.1
Versão SpaceCore: 0.4.2


## 2. Definição de Estados Vetoriais e Operadores Elementares

Definição da base computacional de 1 qubit $\{|0\rangle, |1\rangle\}$, base de superposição $\{|+\rangle, |-\rangle\}$, função de projeção `ketbra(\psi) = |\psi\rangle\langle\psi|` e matrizes de Pauli.


In [2]:
# Qubits da base computacional



phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2.0)
rho_phi_plus = ketbra(phi_plus)

print("Vetor de Bell |Phi+>:")
print(phi_plus)
print("\nMatriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):")
print(rho_phi_plus)

Vetor de Bell |Phi+>:
[0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]

Matriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):
[[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]


## 3. Postulados de Matrizes Densidade (`verifica_densidade`)

Um operador $\rho \in \mathcal{L}(\mathcal{H})$ é uma matriz densidade válida se e somente se satisfazer 3 postulados físicos:
1. **Hermiticidade**: $\rho = \rho^\dagger$
2. **Traço Unitário**: $\text{Tr}(\rho) = 1$
3. **Positividade Semidefinida**: $\rho \ge 0 \iff \lambda_{\min}(\rho) \ge 0$

Testamos a validação sobre o estado de Bell $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.


In [ ]:


resultado_verif = verifica_densidade(rho_phi_plus)
print("Auditoria do estado rho_Phi+:")
for k, v in resultado_verif.items():
    print(f"  {k:12s}: {v}")

## 4. Estrutura do Espaço Composto e Estado GHZ

Para sistemas multipartites $\mathcal{H} = \mathcal{H}_A \otimes \mathcal{H}_B \otimes \mathcal{H}_C$, construímos o produto tensorial sequencial de operadores e geramos o estado maximalmente emaranhado de Greenberger-Horne-Zeilinger (GHZ):

$$|GHZ\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$$


In [ ]:

# Exemplo: Estado |000>
psi_000 = tensor(zero, zero, zero)
print(f"Dimensão do vetor |000>: {psi_000.shape}")
#Teste com o estado GHZ


psi_ghz = ghz(theta=0.0)
rho_ghz = ketbra(psi_ghz)

print(f"Matriz densidade rho_GHZ: formato = {rho_ghz.shape}")
print("Verificação de rho_GHZ:", verifica_densidade(rho_ghz))

## 5. Eliminação de Subsistemas via Traço Parcial (`partial_trace`)

Dada a matriz densidade global $\rho_{ABC}$, calculamos as matrizes reduzidas descartando subsistemas complementares:

$$\rho_{AB} = \text{Tr}_C(\rho_{ABC}) = \sum_k (I_{AB} \otimes \langle k_C|) \rho_{ABC} (I_{AB} \otimes |k_C\rangle)$$

A operação é realizada via contração de tensores com `np.einsum`.


In [ ]:

# Calculando as marginais de dois qubits do estado |GHZ>
rho_AB_ghz = partial_trace(rho_ghz, keep=[0, 1], dims=[2, 2, 2])
rho_AC_ghz = partial_trace(rho_ghz, keep=[0, 2], dims=[2, 2, 2])
rho_BC_ghz = partial_trace(rho_ghz, keep=[1, 2], dims=[2, 2, 2])

print("Marginal rho_AB do estado GHZ (4x4):")
print(rho_AB_ghz)
print("\nAutovalores de rho_AB:", np.round(np.linalg.eigvalsh(rho_AB_ghz), 4))

## 6. Base Ortonormal no Espaço $\text{Herm}(d)$ e Produto de Hilbert-Schmidt

O espaço vetorial real das matrizes Hermitianas $\text{Herm}(d)$ tem dimensão $d^2$. Construímos a base ortonormal $\{E_k\}_{k=1}^{d^2}$ sob o produto interno de Hilbert-Schmidt $\langle A, B \rangle_{\text{HS}} = \text{Tr}(A B) = \delta_{kl}$.

Testamos a ortonormalidade gerando a Matriz de Gram $\text{Tr}(E_i E_j) = I_4$ para $d=2$.


In [ ]:


# Teste com d = 2 (um q-bit)
base_q = base_hermitiana(2)
print(f"Número de matrizes na base de Herm(2): {len(base_q)} (esperado: 2^2 = 4)")

# Teste de ortonormalidade: Matriz de Gram Tr(E_i E_j)
gram = np.array([[float(np.real(np.trace(e1 @ e2))) for e2 in base_q] for e1 in base_q])
print("Matriz de Gram Tr(E_k E_l) para d=2 (deve ser a identidade 4x4):")
print(gram)

## 7. Dualidade do Traço Parcial e Embedding de Operadores Locais (`embed_operator`)

Pela dualidade de Hilbert-Schmidt:

$$\text{Tr}\big[ E_k \text{Tr}_{S^c}(\rho) \big] = \text{Tr}\big[ (E_k \otimes I_{S^c}) \rho \big]$$

Demonstramos numericamente que medir um observável local na marginal equivale a medir o observável incorporado $(E_k \otimes I_{S^c})$ no estado global $\rho$.


In [ ]:

# Demonstração numérica da identidade de dualidade:
# Tr[(O_A (x) I_BC) rho_GHZ] == Tr[O_A Tr_BC(rho_GHZ)]
O_teste = np.array([[2.0, 1.0 - 1.0j], [1.0 + 1.0j, 3.0]], dtype=complex)
M_global_teste = embed_operator(O_teste, sistema=[0], dim=[2, 2, 2])

val_global = np.trace(M_global_teste @ rho_ghz)
val_local = np.trace(O_teste @ partial_trace(rho_ghz, keep=[0], dims=[2, 2, 2]))

print(f"Tr[(O_A (x) I_BC) rho_GHZ] = {val_global.real:.6f} + {val_global.imag:.1e}j")
print(f"Tr[O_A Tr_BC(rho_GHZ)]     = {val_local.real:.6f} + {val_local.imag:.1e}j")
print(f"Diferença absoluta:          {abs(val_global - val_local):.2e}")

## 8. Formatação do SDP Cônico no SDPLab (`build_qmp_sdp`) e Resolução (`solve_qmp`)

Com as igualdades afins escalares $\text{Tr}(M_i \rho) = b_i$, o problema de compatibilidade é convertido no formato cônico padrão do **SDPLab**:

$$\min_{\rho \ge 0} \quad 0 \quad \text{sujeito a} \quad \mathcal{A}(\rho) = b$$

A função `solve_qmp` executa o solver **CLARABEL**, retornando a solução primal $\rho \ge 0$ se factível, ou capturando o certificado de infactibilidade.
